# Panzer — Endpoints publicos

Este notebook muestra como usar `BinancePublicClient` para obtener datos
de mercado de Binance **sin necesidad de API keys**.

Panzer gestiona automaticamente:
- **Rate limiting** sincronizado con Binance (nunca te banean).
- **Calculo de pesos** por endpoint y parametros.
- **Sincronizacion de reloj** con el servidor.

Mercados soportados: `spot`, `um` (USDT-M Futures), `cm` (COIN-M Futures).

## 1. Crear un cliente

In [1]:
from panzer import BinancePublicClient

# Al crear el cliente se descarga /exchangeInfo para obtener los limites de peso.
# safety_ratio=0.9 significa que dormira al llegar al 90% del limite (ej: 5400/6000).
client = BinancePublicClient(market="spot", safety_ratio=0.9)

print(f"Mercado:     {client.market}")
print(f"Base URL:    {client.base_url}")
print(f"Max peso/min: {client.limiter.max_per_minute}")

2026-03-08 12:17:51     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-08 12:17:52     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90
2026-03-08 12:17:52     INFO [panzer.binance.public.spot] Reloj sincronizado: offset=-114 ms


Mercado:     spot
Base URL:    https://api.binance.com
Max peso/min: 6000


## 2. Sincronizar reloj

Antes de hacer muchas peticiones es recomendable sincronizar el reloj local
con el del servidor. Esto mejora la precision del rate limiter.

In [2]:
# Llama a /time 3 veces para estimar el desfase local vs servidor
client.ensure_time_offset_ready(min_samples=3)

offset_ms = client.time_offset.current_offset() * 1000
print(f"Desfase estimado: {offset_ms:.1f} ms")
print(f"Hora servidor (estimada): {client.now_server_ms()} ms")

Desfase estimado: -114.4 ms
Hora servidor (estimada): 1772968673117 ms


## 3. Klines (velas)

Datos OHLCV. Cada vela es una lista de 12 elementos:
`[open_time, open, high, low, close, volume, close_time, quote_vol, trades, taker_buy_base, taker_buy_quote, ignore]`

In [3]:
klines = client.klines("BTCUSDT", "1h", limit=5)

print(f"Obtenidas {len(klines)} velas de 1h para BTCUSDT\n")
for k in klines:
    open_price, high, low, close = k[1], k[2], k[3], k[4]
    volume = k[5]
    trades = k[8]
    print(f"  O={open_price:>12s}  H={high:>12s}  L={low:>12s}  C={close:>12s}  Vol={volume:>14s}  Trades={trades}")

Obtenidas 5 velas de 1h para BTCUSDT

  O=67161.52000000  H=67465.11000000  L=67154.91000000  C=67319.91000000  Vol= 1010.57902000  Trades=103709
  O=67319.90000000  H=67577.36000000  L=67156.93000000  C=67539.33000000  Vol= 1008.62460000  Trades=97378
  O=67539.34000000  H=67905.03000000  L=67474.00000000  C=67681.00000000  Vol= 1111.87857000  Trades=143975
  O=67681.00000000  H=68200.00000000  L=67680.99000000  C=67904.14000000  Vol=  646.22527000  Trades=116368
  O=67904.15000000  H=67904.15000000  L=67456.88000000  C=67608.15000000  Vol=  193.29863000  Trades=46817


### Intervalos disponibles

`1s`, `1m`, `3m`, `5m`, `15m`, `30m`, `1h`, `2h`, `4h`, `6h`, `8h`, `12h`, `1d`, `3d`, `1w`, `1M`

In [4]:
# Klines con rango temporal (timestamps en milisegundos)
import time

end_ms = int(time.time() * 1000)
start_ms = end_ms - 24 * 60 * 60 * 1000  # ultimas 24h

klines_24h = client.klines("ETHUSDT", "15m", start_time=start_ms, end_time=end_ms, limit=1000)
print(f"Velas de 15m en las ultimas 24h: {len(klines_24h)}")

Velas de 15m en las ultimas 24h: 96


### klines_range — paginacion automatica

`klines()` devuelve como maximo 1000 velas por peticion (limite de Binance).
Si el rango temporal implica mas de 1000 velas, **se pierden silenciosamente**.

`klines_range()` resuelve esto: divide el rango en bloques de 1000 velas,
lanza las peticiones en paralelo y deduplica por open timestamp.

In [5]:
import time

end_ms = int(time.time() * 1000)
start_ms = end_ms - 7 * 24 * 60 * 60 * 1000  # ultimos 7 dias

# klines() normal: Binance trunca a 1000 velas maximo
klines_truncated = client.klines("BTCUSDT", "1m", start_time=start_ms, end_time=end_ms, limit=1000)
print(f"klines() con limit=1000:  {len(klines_truncated)} velas  (truncado por Binance)")

# klines_range(): obtiene TODAS las velas del rango, paginando en paralelo
klines_full = client.klines_range("BTCUSDT", "1m", start_time=start_ms, end_time=end_ms)
print(f"klines_range():           {len(klines_full)} velas  (rango completo)")

# Verificar continuidad: no hay huecos entre velas consecutivas
gaps = 0
for i in range(1, len(klines_full)):
    expected = klines_full[i - 1][0] + 60_000  # open_time + 1 minuto
    if klines_full[i][0] != expected:
        gaps += 1
print(f"Huecos detectados:        {gaps}")

2026-03-08 12:17:54     INFO [panzer.binance.public.spot] klines_range: 11 peticiones en paralelo para BTCUSDT 1m


klines() con limit=1000:  1000 velas  (truncado por Binance)


2026-03-08 12:17:55     INFO [panzer.binance.public.spot] parallel_get: 11 jobs, 1 lotes, used_local=51
2026-03-08 12:17:55     INFO [panzer.binance.public.spot] klines_range: 10080 velas obtenidas para BTCUSDT 1m


klines_range():           10080 velas  (rango completo)
Huecos detectados:        0


## 4. Trades recientes

In [6]:
trades = client.trades("BTCUSDT", limit=5)

print(f"Ultimos {len(trades)} trades de BTCUSDT:\n")
for t in trades:
    side = "SELL" if t["isBuyerMaker"] else "BUY "
    print(f"  id={t['id']}  {side}  price={t['price']:>12s}  qty={t['qty']:>12s}")

Ultimos 5 trades de BTCUSDT:

  id=6078727827  BUY   price=67608.15000000  qty=  0.00041000
  id=6078727828  BUY   price=67608.15000000  qty=  0.01000000
  id=6078727829  BUY   price=67608.15000000  qty=  0.00172000
  id=6078727830  BUY   price=67608.15000000  qty=  0.00126000
  id=6078727831  BUY   price=67608.15000000  qty=  0.00124000


## 5. Trades agregados (aggTrades)

Los aggTrades comprimen varios trades atomicos en uno cuando comparten precio y lado.
Cada aggTrade tiene un rango de trade IDs (`f` a `l`).

In [7]:
agg = client.agg_trades("BTCUSDT", limit=5)

print(f"Ultimos {len(agg)} aggTrades:\n")
for t in agg:
    maker = "maker" if t["m"] else "taker"
    atomic_count = t["l"] - t["f"] + 1
    print(f"  aggId={t['a']}  price={t['p']:>12s}  qty={t['q']:>10s}  {maker}  ({atomic_count} trades atomicos)")

Ultimos 5 aggTrades:

  aggId=3896512068  price=67608.15000000  qty=0.00041000  taker  (1 trades atomicos)
  aggId=3896512069  price=67608.15000000  qty=0.01000000  taker  (1 trades atomicos)
  aggId=3896512070  price=67608.15000000  qty=0.00172000  taker  (1 trades atomicos)
  aggId=3896512071  price=67608.15000000  qty=0.00126000  taker  (1 trades atomicos)
  aggId=3896512072  price=67608.15000000  qty=0.00124000  taker  (1 trades atomicos)


## 6. Order book (depth)

In [8]:
book = client.depth("BTCUSDT", limit=5)

best_bid = float(book["bids"][0][0])
best_ask = float(book["asks"][0][0])
spread = best_ask - best_bid

print(f"Order book BTCUSDT (top 5 niveles)\n")
print(f"  {'BIDS (compra)':>30s}  |  {'ASKS (venta)':<30s}")
print(f"  {'price':>14s} {'qty':>14s}  |  {'price':<14s} {'qty':<14s}")
print(f"  {'-'*30}  |  {'-'*30}")
for i in range(5):
    bid_p, bid_q = book["bids"][i]
    ask_p, ask_q = book["asks"][i]
    print(f"  {bid_p:>14s} {bid_q:>14s}  |  {ask_p:<14s} {ask_q:<14s}")

print(f"\n  Best bid: {best_bid}  |  Best ask: {best_ask}  |  Spread: {spread:.2f}")

Order book BTCUSDT (top 5 niveles)

                   BIDS (compra)  |  ASKS (venta)                  
           price            qty  |  price          qty           
  ------------------------------  |  ------------------------------
  67608.14000000     0.70897000  |  67608.15000000 0.00376000    
  67608.13000000     0.00016000  |  67608.16000000 0.00080000    
  67608.07000000     0.00761000  |  67608.22000000 0.00008000    
  67607.71000000     0.02815000  |  67608.47000000 0.00016000    
  67607.30000000     0.00200000  |  67608.48000000 0.00016000    

  Best bid: 67608.14  |  Best ask: 67608.15  |  Spread: 0.01


## 7. Exchange info

Metadata del exchange: simbolos disponibles, filtros de precio/cantidad, etc.

In [9]:
# Info de un simbolo concreto
info = client.exchange_info(symbol="BTCUSDT")
sym = info["symbols"][0]

print(f"Simbolo:   {sym['symbol']}")
print(f"Status:    {sym['status']}")
print(f"Base:      {sym['baseAsset']}")
print(f"Quote:     {sym['quoteAsset']}")
print(f"Filtros:   {len(sym['filters'])}")
for f in sym["filters"]:
    print(f"  - {f['filterType']}")

Simbolo:   BTCUSDT
Status:    TRADING
Base:      BTC
Quote:     USDT
Filtros:   11
  - PRICE_FILTER
  - LOT_SIZE
  - ICEBERG_PARTS
  - MARKET_LOT_SIZE
  - TRAILING_DELTA
  - PERCENT_PRICE_BY_SIDE
  - NOTIONAL
  - MAX_NUM_ORDERS
  - MAX_NUM_ORDER_LISTS
  - MAX_NUM_ALGO_ORDERS
  - MAX_NUM_ORDER_AMENDS


## 8. Multiples mercados

El mismo API funciona para Spot, USDT-M Futures y COIN-M Futures.
Solo cambia el parametro `market`.

In [10]:
# Comparar precio de BTC en los 3 mercados
for market in ["spot", "um", "cm"]:
    c = BinancePublicClient(market=market)
    symbol = "BTCUSDT" if market != "cm" else "BTCUSD_PERP"
    kline = c.klines(symbol, "1m", limit=1)
    close = kline[0][4]
    print(f"  {market:>4s}  {symbol:<16s}  close={close}")

2026-03-08 12:17:56     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-08 12:17:57     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90
2026-03-08 12:17:58     INFO [panzer.binance.public.spot] Reloj sincronizado: offset=-114 ms
2026-03-08 12:17:58     INFO [panzer.binance.public.um] Inicializando BinancePublicClient(market=um)


  spot  BTCUSDT           close=67612.66000000


2026-03-08 12:17:59     INFO [panzer.binance.public.um] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.90
2026-03-08 12:17:59     INFO [panzer.binance.public.um] Reloj sincronizado: offset=-115 ms
2026-03-08 12:18:00     INFO [panzer.binance.public.cm] Inicializando BinancePublicClient(market=cm)


    um  BTCUSDT           close=67575.10


2026-03-08 12:18:00     INFO [panzer.binance.public.cm] Rate limiter inicializado: max_per_minute=2400 safety_ratio=0.90
2026-03-08 12:18:03     INFO [panzer.binance.public.cm] Reloj sincronizado: offset=-119 ms


    cm  BTCUSD_PERP       close=67576.6


## 9. Endpoint generico con `get()`

Para cualquier endpoint publico que no tenga wrapper, puedes usar `get()` directamente.
El peso se calcula automaticamente o puedes pasarlo manualmente.

In [11]:
# Ticker 24h — no tiene wrapper dedicado
ticker = client.get("/api/v3/ticker/24hr", params={"symbol": "BTCUSDT"})

print(f"BTCUSDT 24h:")
print(f"  Precio actual:   {ticker['lastPrice']}")
print(f"  Cambio 24h:      {ticker['priceChangePercent']}%")
print(f"  Volumen 24h:     {ticker['volume']} BTC")
print(f"  Max 24h:         {ticker['highPrice']}")
print(f"  Min 24h:         {ticker['lowPrice']}")

BTCUSDT 24h:
  Precio actual:   67626.86000000
  Cambio 24h:      -0.747%
  Volumen 24h:     16048.63888000 BTC
  Max 24h:         68227.28000000
  Min 24h:         66547.15000000


## 10. Rate limiter — inspeccion en tiempo real

Puedes consultar el estado del limiter en cualquier momento para ver
cuanto peso has consumido en la ventana actual.

In [12]:
lim = client.limiter

print(f"Peso usado (local):      {lim.used_local}")
print(f"Peso usado (servidor):   {lim.last_server_used}")
print(f"Limite por minuto:       {lim.max_per_minute}")
print(f"Ratio de seguridad:      {lim.safety_ratio}")
print(f"Limite efectivo:         {int(lim.max_per_minute * lim.safety_ratio)}")
print(f"Margen restante:         {int(lim.max_per_minute * lim.safety_ratio) - lim.used_local}")

Peso usado (local):      2
Peso usado (servidor):   2
Limite por minuto:       6000
Ratio de seguridad:      0.9
Limite efectivo:         5400
Margen restante:         5398


## 11. Manejo de errores

In [13]:
from panzer.errors import BinanceAPIException

try:
    client.klines("SIMBOLO_INVALIDO", "1m")
except BinanceAPIException as e:
    print(f"Error capturado:")
    print(f"  HTTP status:  {e.status_code}")
    print(f"  Binance code: {e.error_payload.code}")
    print(f"  Mensaje:      {e.error_payload.msg}")
    print(f"  URL:          {e.url}")

2026-03-08 12:18:04    ERROR [panzer.errors] BinanceAPIException raised: status=400 method=GET url=https://api.binance.com/api/v3/klines?symbol=SIMBOLO_INVALIDO&interval=1m&limit=500 code=-1121 msg=Invalid symbol.


Error capturado:
  HTTP status:  400
  Binance code: -1121
  Mensaje:      Invalid symbol.
  URL:          https://api.binance.com/api/v3/klines?symbol=SIMBOLO_INVALIDO&interval=1m&limit=500
